In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Base Feature Distribution Visualization Utilities (`plots/plot_utils.ipynb`)

This notebook generates visualization plots strictly for **Configured Base Features** specified in `config/triage_conf.json`.

### Visualizations Generated:
- **Target Class Distribution (`esi`)**: Bar chart of patient counts across ESI triage levels.
- **Configured Base Feature Density Plots**: Faceted KDE density plots by ESI level.
- **Configured Base Feature Boxplots**: Faceted boxplots highlighting medians and IQRs across ESI levels.
- **Saved Artifacts**: PNG plots exported to `plots/image/` and summary CSV tables exported to `plots/csv/`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(ggplot2)
library(dplyr)
library(tidyr)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Features Count:  ", length(config$features$data_name), "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Dataset & Ensure Output Directories
# ---------------------------------------------------------
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]

raw_df <- get(data_obj_name, envir = data_env)

target_col   <- config$classes$target_col
feature_cols <- config$features$data_name
target_classes <- as.character(config$classes$outputs)

selected_cols <- intersect(c(feature_cols, target_col), names(raw_df))
df <- raw_df[, selected_cols, drop = FALSE]

df[[target_col]] <- factor(df[[target_col]], levels = target_classes)

# Output directories
img_dir <- "../plots/image"
if (!dir.exists(img_dir)) img_dir <- "image"
if (!dir.exists(img_dir)) dir.create(img_dir, recursive = TRUE)

csv_dir <- "../plots/csv"
if (!dir.exists(csv_dir)) csv_dir <- "csv"
if (!dir.exists(csv_dir)) dir.create(csv_dir, recursive = TRUE)

cat(sprintf("Loaded dataset: %d rows x %d cols (Features in JSON: %d)\n", nrow(df), ncol(df), length(feature_cols)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Target Class (esi) Distribution Plot
# ---------------------------------------------------------
target_summary <- df %>%
  group_by(.data[[target_col]]) %>%
  summarise(Count = n(), .groups = "drop") %>%
  mutate(Percentage = (Count / sum(Count)) * 100)

cat("=== ESI Target Class Summary ===\n")
print(target_summary)
write.csv(target_summary, file.path(csv_dir, "esi_target_distribution.csv"), row.names = FALSE)

p_target <- ggplot(target_summary, aes(x = .data[[target_col]], y = Count, fill = .data[[target_col]])) +
  geom_bar(stat = "identity", color = "black", alpha = 0.85) +
  geom_text(aes(label = sprintf("%s\n(%.1f%%)", format(Count, big.mark=","), Percentage)), vjust = -0.2, size = 3.8) +
  scale_fill_brewer(palette = "Set1") +
  theme_minimal(base_size = 13) +
  labs(
    title = "Target Distribution: Emergency Severity Index (ESI)",
    subtitle = "Patient frequency across ESI triage levels 1 to 5",
    x = "ESI Triage Level",
    y = "Patient Count",
    fill = "ESI Level"
  ) +
  theme(legend.position = "none", panel.grid.minor = element_blank())

print(p_target)
ggsave(file.path(img_dir, "target_esi_distribution.png"), plot = p_target, width = 8, height = 5)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Memory-Safe Base Features Density & Boxplot Distributions
# ---------------------------------------------------------
num_cols <- names(df)[sapply(df, is.numeric)]
cat("Plotting distributions for base features in config:", paste(num_cols, collapse = ", "), "\n")

# Summary statistics computed on FULL dataset
num_summary <- df %>%
  pivot_longer(cols = all_of(num_cols), names_to = "Feature", values_to = "Value") %>%
  group_by(Feature) %>%
  summarise(
    Mean = round(mean(Value, na.rm = TRUE), 2),
    SD = round(sd(Value, na.rm = TRUE), 2),
    Median = round(median(Value, na.rm = TRUE), 2),
    IQR = round(IQR(Value, na.rm = TRUE), 2),
    Min = round(min(Value, na.rm = TRUE), 2),
    Max = round(max(Value, na.rm = TRUE), 2),
    .groups = "drop"
  )

write.csv(num_summary, file.path(csv_dir, "numeric_features_summary.csv"), row.names = FALSE)

# Representative sample (10,000 rows) for memory-safe ggplot rendering
set.seed(config$training$random_state)
sample_size <- min(nrow(df), 10000)
df_sample <- df[sample(seq_len(nrow(df)), sample_size), ]

df_sample_long <- df_sample %>%
  pivot_longer(cols = all_of(num_cols), names_to = "Feature", values_to = "Value")

p_density <- ggplot(df_sample_long, aes(x = Value, fill = .data[[target_col]])) +
  geom_density(alpha = 0.45) +
  facet_wrap(~ Feature, scales = "free", ncol = 3) +
  scale_fill_brewer(palette = "Set1") +
  theme_minimal(base_size = 12) +
  labs(
    title = "Base Feature Density Distributions Grouped by ESI Class",
    subtitle = sprintf("KDE density plots for numerical features (Stratified sample N = %s)", format(sample_size, big.mark=",")),
    x = "Feature Value",
    y = "Density",
    fill = "ESI Level"
  ) +
  theme(legend.position = "bottom", panel.grid.minor = element_blank())

print(p_density)
ggsave(file.path(img_dir, "numeric_features_density.png"), plot = p_density, width = 12, height = 8)

p_box <- ggplot(df_sample_long, aes(x = .data[[target_col]], y = Value, fill = .data[[target_col]])) +
  geom_boxplot(outlier.size = 0.5, outlier.alpha = 0.3, alpha = 0.8) +
  facet_wrap(~ Feature, scales = "free_y", ncol = 3) +
  scale_fill_brewer(palette = "Set1") +
  theme_minimal(base_size = 12) +
  labs(
    title = "Base Feature Distributions across ESI Triage Levels",
    subtitle = sprintf("Boxplots for numerical features (Stratified sample N = %s)", format(sample_size, big.mark=",")),
    x = "ESI Level",
    y = "Feature Value",
    fill = "ESI Level"
  ) +
  theme(legend.position = "none", panel.grid.minor = element_blank())

print(p_box)
ggsave(file.path(img_dir, "numeric_features_boxplots.png"), plot = p_box, width = 12, height = 8)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Summary Report of Generated Artifacts
# ---------------------------------------------------------
cat("=== Base Feature Distribution Plotting Complete ===\n")
cat("Generated image plots saved to:\n")
cat("  - Target Distribution:     ", file.path(img_dir, "target_esi_distribution.png"), "\n")
cat("  - JSON Numerical Density:  ", file.path(img_dir, "numeric_features_density.png"), "\n")
cat("  - JSON Numerical Boxplots: ", file.path(img_dir, "numeric_features_boxplots.png"), "\n")
cat("Generated CSV summaries saved to:\n")
cat("  - ESI Target Summary CSV:  ", file.path(csv_dir, "esi_target_distribution.csv"), "\n")
cat("  - JSON Numeric Summary CSV:", file.path(csv_dir, "numeric_features_summary.csv"), "\n")